# L9b: Black–Scholes–Merton Premiums and Pricing American Options using the CRR Binomial Model
Previously, we priced European options by Monte Carlo simulation. Today, we price them with the Black–Scholes–Merton formulas. However, standard options on stocks and ETFs are American-style and can be exercised at any time. What is the right to exercise early worth?

> __Learning Objectives:__
> 
> By the end of this lecture, you should be able to:
> * __Price European options with the Black–Scholes–Merton model:__ Compute European call and put premiums with the Black–Scholes–Merton formulas and back out the implied volatility from a market premium.
> * __Distinguish between European and American options:__ Understand the key differences in exercise timing and pricing implications between European options (exercisable only at expiration) and American options (exercisable at any time before or on expiration).
> * __Apply the binomial model to price American options:__ Use the CRR model to construct binomial trees and compute option premiums, incorporating the decision rule for early exercise.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L9b) lists this lecture's symbols with their meanings and units.

Let's get started!
___

## Examples

__We will work through these examples in lecture:__

> [▶ Let's compute the BSM premium for European call and put options](../L9a/CHEME-5660-L9a-Example-BSM-Premium-Fall-2026.ipynb). In this example, we compute European call and put premiums by Monte Carlo simulation and with the Black–Scholes–Merton formulas. We then look at how the premium changes with strike price, in the model and in real S&P 500 index option quotes.

> [▶ Let's compute the Premium for Market-Quoted American Call and Put Options using CRR](CHEME-5660-L9b-Example-AmericanContractsPremium-Fall-2026.ipynb). In this example, we'll compute the price of American call and put option contracts using the CRR binomial model, compare our results to observed market prices, and measure each contract's early-exercise premium.

__Review this example on your own.__

> [▶ Let's compute the Premium for Hull's Hypothetical American Put using a binomial lattice](CHEME-5660-L9b-Example-Hull-American-Put-Fall-2026.ipynb). In this example, we'll compute the price of a hypothetical American put option contract using backward induction over a binomial lattice model, and compare our results to the same contract written as a European contract.
___

## Concept Review: A Monte Carlo Premium
How do we compute the premium of a European option? Previously, we found it by setting the contract's expected net present value to zero under the risk-neutral pricing rule $\mathbb{Q}$. Under $\mathbb{Q}$, the expected share price grows at the benchmark risk-free rate $g_{y}$. For a call or a put with payoff $V_{\star}(K,S(T))$, the premium of a European-style contract is the discounted expected payoff under $\mathbb{Q}$:
$$
\boxed{
\begin{align*}
\mathcal{P}_{\star}^{\text{EU}} = \mathcal{D}^{-1}_{T,0}(g_{y})\cdot\mathbb{E}_{\mathbb{Q}}\left[V_{\star}(K,S(T))\right]\quad\text{where }\star\in\{c,p\}
\end{align*}}
$$
where $K$ is the strike price, $S(T)$ is the share price at expiration $T$ (units: years), and $\mathcal{D}^{-1}_{T,0}(g_{y})=e^{-g_{y}T}$ discounts the payoff back to today.

Simulation gives a direct way to estimate the market premium. The payoff depends only on the share price at expiration, so we can draw $S(T)$ directly from the solution of geometric Brownian motion, with drift $g_{y}$. Let's outline some pseudocode for the Monte Carlo premium of a European contract.

__Initialize__: Given the share price $S(0)$, the strike price $K$, the payoff function $V(K,S)$ of the call or put, the time to expiration $T$ (units: years), the volatility $\sigma$, the risk-free rate $g_{y}$, and the number of simulated paths $M$.

1. For each path $i = 1,2,\dots,M$ __do__:
   - a. Draw a standard normal random number $Z_{i}$.
   - b. Compute the share price at expiration: $S_{i}(T) \gets S(0)\cdot\exp\Bigl[\left(g_{y}-\frac{\sigma^{2}}{2}\right)T + \sigma\sqrt{T}\,Z_{i}\Bigr]$.
   - c. Compute the payoff: $V_{i} \gets V(K,S_{i}(T))$.
2. The premium is the discounted average payoff: $\mathcal{P} \gets \mathcal{D}^{-1}_{T,0}(g_{y})\cdot\frac{1}{M}\sum_{i=1}^{M}V_{i}$.

There is an alternative way to think about the premium, which the [contingent claims notebook](advanced/contingent_claims/CHEME-5660-L9b-Advanced-ContingentClaims-BSM-Fall-2026.ipynb) works through.

Let's estimate the premium of a European call and put by simulation (Approach 1 in Tasks 1 and 2):

> __Example:__
>
> [▶ Let's compute the BSM premium for European call and put options](../L9a/CHEME-5660-L9a-Example-BSM-Premium-Fall-2026.ipynb). In this example, we compute European call and put premiums by Monte Carlo simulation and with the Black–Scholes–Merton formulas. We then look at how the premium changes with strike price, in the model and in real S&P 500 index option quotes.

The Monte Carlo premium is an estimate: it changes with the random draws, and it becomes more precise as the number of paths $M$ grows.
___

## Company Profile: The Options Clearing Corporation

[The Options Clearing Corporation (OCC)](https://www.theocc.com/) stands between the buyer and the seller of every U.S. listed option. OCC does not deal with individual investors directly. It deals with its clearing members, brokerage firms that settle option trades with OCC for themselves and for their customers. OCC opened on April 26, 1973, as CBOE Clearing Corporation, a subsidiary of the new Chicago Board Options Exchange. In 1975, it became OCC, an [industry clearinghouse for all U.S. equity options](https://annualreport.theocc.com/2023/celebrating-50-years), headquartered in Chicago.

> __What does the firm do?__
>
> * __Exercise and assignment:__ When a holder exercises an American option early, the notice goes to OCC through the holder's clearing member. OCC then [picks at random](https://www.optionseducation.org/referencelibrary/faq/options-assignment) a clearing member whose customers have sold the same contract, and that firm chooses one of those customers. The chosen seller is assigned: they must now buy the shares at the strike price (a put) or sell them at the strike price (a call).
> * __Automatic exercise at expiration:__ OCC introduced exercise by exception in 1977, the year it began clearing puts. Under OCC Rule 805, an expiring equity option that is [in the money by 0.01 USD/share or more](https://www.optionseducation.org/referencelibrary/faq/options-exercise) is exercised unless the holder's clearing member instructs otherwise.
> * __Monte Carlo collateral and binomial trees:__ OCC decides how much collateral each clearing member must post with [STANS](https://www.theocc.com/risk-management/margin-methodology), a large-scale Monte Carlo system in use since August 2006. If a clearing member fails, OCC expects to need about two trading days to sell off that member's positions, and prices can move against OCC during that time. So STANS simulates 10,000 scenarios of how share prices and implied volatilities could change over two days, reprices every contract in each scenario, and sets the collateral to cover the losses in the bad scenarios. American options are [repriced on a binomial tree](https://www.govinfo.gov/content/pkg/FR-2019-07-09/pdf/2019-14627.pdf), because the Black–Scholes–Merton formula cannot value early exercise.

__Explore further:__

* __Jobs and internships:__ Browse OCC's [careers page](https://www.theocc.com/careers) and its [internship program](https://www.theocc.com/careers/gain-experience-with-occ) in Chicago and Dallas/Fort Worth. The summer 2027 postings include an internship in Quantitative Risk Management, the group that builds OCC's margin models.
* __Videos:__ The [Options Industry Council](https://www.optionseducation.org/), which OCC formed with the exchanges in 1992, publishes a [YouTube channel](https://www.youtube.com/@OptionsEdu) on listed options. Start with [Option Exercise and Assignment](https://www.youtube.com/embed/OC-gkRp9hlw) and [The Automatic Exercise Process Explained](https://www.youtube.com/embed/cFckMpLdXPE).

__Connection to today's lecture:__ The seller of an American put can be forced to buy the shares at the strike price on any day the holder chooses to exercise. OCC holds collateral from the seller to cover that obligation. An American put is worth more than the same European put, and the difference, the early-exercise premium, is largest for puts deep in the money, because the European holder must wait until expiration to collect the strike price and gives up the interest on it. If OCC priced the put as European, it would hold too little collateral. That is why it prices American options on a binomial tree.
___

## Black–Scholes–Merton (BSM) Premiums
The price of a European option contract can also be determined using the Black–Scholes–Merton (BSM) model.

> __Nobel Prize 1997:__ The 1997 Nobel Prize in Economic Sciences was awarded to Robert C. Merton and Myron S. Scholes for their work in developing the Black–Scholes–Merton model. Fischer Black was also a key contributor to the model but had passed away before the prize was awarded. [See more here](https://www.nobelprize.org/prizes/economic-sciences/1997/summary/).

We assume that the price of a non-dividend-paying share follows geometric Brownian motion with constant volatility $\sigma$, and that the continuously compounded risk-free rate $g_y$ is constant.

__The BSM equation.__ Write the premium at time $t$, when the share price is $S$, as $\mathcal{P}(S,t)$. At expiration, it equals the payoff, $\mathcal{P}(S,T)=V(K,S)$. Under the BSM assumptions, the premium satisfies the __BSM equation__:
$$
\boxed{
\begin{align*}
\frac{\partial\mathcal{P}}{\partial t}+g_{y}\,S\,\frac{\partial\mathcal{P}}{\partial S}+\frac{1}{2}\,\sigma^{2}S^{2}\,\frac{\partial^{2}\mathcal{P}}{\partial S^{2}} = g_{y}\,\mathcal{P}
\end{align*}}
$$
Solving it backward in time from the payoff gives today's premium, $\mathcal{P}(S(0),0)$, and this solution equals the risk-neutral premium $\mathcal{P}^{\text{EU}}$ that the Monte Carlo simulation estimates. Instead of simulating, we can compute the premium with [the Black–Scholes analytical solution](https://en.wikipedia.org/wiki/Black–Scholes_model) for a European call and put option ($t = 0$):
$$
\boxed{
\begin{align*}
\mathcal{P}_{c}(K,S(0)) & = N(d_{+})\cdot{S}(0) - N(d_{-})\cdot{K}\cdot\mathcal{D}^{-1}_{T,0}(g_{y})\\
\mathcal{P}_{p}(K,S(0)) & = N(-d_{-})\cdot{K}\cdot\mathcal{D}^{-1}_{T,0}(g_{y}) - N(-d_{+})\cdot{S}(0)
\end{align*}}
$$
where:
$$
\begin{align*}
d_{+} & = \frac{1}{\sigma\sqrt{T}}\left[\ln\left(\frac{S(0)}{K}\right) + \left(g_{y}+\frac{\sigma^{2}}{2}\right)T\right] \\
d_{-} & = d_{+} - \sigma\sqrt{T}
\end{align*}
$$
and $N(\dots)$ denotes the standard normal cumulative distribution function, $\sigma$ is the volatility of the underlying asset, $g_{y}$ is the risk-free rate, and $T$ is the time to expiration of the contract in years.

The [contingent claims notebook](advanced/contingent_claims/CHEME-5660-L9b-Advanced-ContingentClaims-BSM-Fall-2026.ipynb) derives the BSM equation and shows that its solution is the risk-neutral premium.

__Which volatility $\sigma$ should we use?__ If we knew the right volatility, the BSM model (or the Monte Carlo model) would compute the observed market premium. However, in practice we observe the market premium and want to back out the volatility that makes the model match the market. This is called __implied volatility__:

> __Historical or implied volatility?__ There are two ways to put a number on $\sigma$, and the second also gives a quick range for the share price:
> * __Historical volatility:__ Previously, we estimated the volatility $\sigma$ from past share prices as $\sigma_{\mathrm{ann}}=\sqrt{\Delta t}\cdot\sigma_{g}$, where $\sigma_{g}$ is the standard deviation of the growth rates (units: $\mathrm{yr}^{-1}$) and $\Delta t$ is the time between price observations (units: years). This estimate looks backward.
> * __Implied volatility:__ The [implied volatility](https://en.wikipedia.org/wiki/Implied_volatility) (IV) is the market's forecast of a likely future movement in the share price. The implied volatility is often equal to or larger than the historical volatility. It is a forward-looking measure of uncertainty. 
> * __How can we use it?__ The implied volatility represents a one-standard-deviation movement in the share price over the next year. Over a horizon of $T$ years, the size of that move scales with $\sqrt{T}$.
>
> Moving one standard deviation up or down in the solution of geometric Brownian motion gives a back-of-the-envelope range for the share price:
> $$
> \begin{align*}
> S(T) & = S(0)\exp\left[\mu_{g}T + \sigma\sqrt{T}\,Z\right] && \text{solution of geometric Brownian motion}\\
> S_{\pm} & = S(0)\exp\left[\mu_{g}T \pm \text{IV}\sqrt{T}\right] && \text{set } \sigma=\text{IV} \text{ and } Z=\pm 1\\
> & \approx S(0)\exp\left[\pm\,\text{IV}\sqrt{T}\right] && \mu_{g}T \text{ is small next to } \text{IV}\sqrt{T} \text{ for short } T\\
> & \approx S(0)\left(1 \pm \text{IV}\sqrt{T}\right) && e^{x}\approx 1+x \text{ for small } x
> \end{align*}
> $$
> where $\mu_{g}$ is the mean growth rate, $Z$ is a standard normal random variable, and $S_{\pm}$ is the share price one standard deviation above or below $S(0)$. Under the model, the share price ends inside this range about 68% of the time.

Let's outline some pseudocode for backing out the implied volatility by bisection.

__Initialize__: Given the market premium $\mathcal{P}^{\text{market}}$, the share price $S(0)$, the strike price $K$, the time to expiration $T$ (units: years), the risk-free rate $g_{y}$, a volatility bracket $\sigma_{\text{lo}}<\sigma_{\text{hi}}$ whose BSM premiums satisfy $\mathcal{P}(\sigma_{\text{lo}})\leq\mathcal{P}^{\text{market}}\leq\mathcal{P}(\sigma_{\text{hi}})$, a tolerance $\epsilon>0$, a maximum number of iterations $\texttt{maxiter}$, and an iteration counter $i\gets 1$.

While $\sigma_{\text{hi}}-\sigma_{\text{lo}}>\epsilon$ and $i\leq\texttt{maxiter}$ __do__:
1. Compute the midpoint $\sigma_{\text{mid}}\gets(\sigma_{\text{lo}}+\sigma_{\text{hi}})/2$ and its BSM premium $\mathcal{P}(\sigma_{\text{mid}})$.
2. Compare $\mathcal{P}(\sigma_{\text{mid}})$ with $\mathcal{P}^{\text{market}}$. The BSM premium rises with $\sigma$, so:
   * If $\mathcal{P}(\sigma_{\text{mid}})<\mathcal{P}^{\text{market}}$, the midpoint volatility is too low. The implied volatility is above $\sigma_{\text{mid}}$, so move the lower end up: $\sigma_{\text{lo}}\gets\sigma_{\text{mid}}$.
   * If $\mathcal{P}(\sigma_{\text{mid}})\geq\mathcal{P}^{\text{market}}$, the midpoint volatility is too high or exactly right. The implied volatility is at or below $\sigma_{\text{mid}}$, so move the upper end down: $\sigma_{\text{hi}}\gets\sigma_{\text{mid}}$.
3. Update the iteration counter $i\gets i+1$.

The implied volatility is the midpoint of the final bracket, $\hat{\sigma}\gets(\sigma_{\text{lo}}+\sigma_{\text{hi}})/2$. The BSM premium rises with $\sigma$, so exactly one $\sigma$ matches the market premium, and each step halves the bracket around it. 

> __Advanced example__: The [SPXW volatility skew notebook](advanced/spxw_volatility_skew/CHEME-5660-L9b-Advanced-SPXW-Volatility-Skew-Fall-2026.ipynb) applies this bisection method to a chain of S&P 500 index options. Each strike and expiration has its own implied volatility, although the BSM model assumes one constant $\sigma$ (this is what is known as the "volatility smile" or "volatility skew"). 
> 
> Alternatively, to price a contract without a quote, or on a future date, we can predict its implied volatility from a surface fitted to today's quotes across strikes and expirations ([Sun et al., 2026](https://arxiv.org/abs/2605.13998)).

__Interesting fact:__ The call and put premiums are linked. For a European call and put with the same strike and expiration, [put–call parity](CHEME-5660-L9b-InterestingFact-PutCallParity-Fall-2026.ipynb) gives $\mathcal{P}_{c}(K,S(0))-\mathcal{P}_{p}(K,S(0))=S(0)-K\cdot\mathcal{D}^{-1}_{T,0}(g_{y})$. It holds only for European contracts.

__Review on your own:__ In the [example](../L9a/CHEME-5660-L9a-Example-BSM-Premium-Fall-2026.ipynb), Approach 2 in Tasks 1 and 2 computes the same premiums with the Black–Scholes–Merton formulas, and Task 3 compares them with real S&P 500 index option quotes.

The BSM formulas cannot value early exercise, so let's look at how to price American options.
___

## American Style Options Contracts
American-style options contracts allow the holder to exercise the option __at any time__ before or on the expiration date. We write the premium of an American-style contract as:
$$
\begin{align*}
\mathcal{P}_{\star}^{\text{AM}} = \max_{\tau\leq T} \mathbb{E}_{\mathbb{Q}}\Bigl(\mathcal{D}^{-1}_{\tau,0}(g_{y})\cdot V_{\star}(K,S(\tau))\Bigr) \geq \mathcal{P}_{\star}^{\text{EU}}\qquad\text{for }\star\in\{c,p\}
\end{align*}
$$
where $\mathcal{D}^{-1}_{\tau,0}(g_{y})$ discounts the payoff $V_{\star}(K,S(\tau))$ from the exercise time $\tau$ back to today, and $\tau$ is chosen using only the information available at that time (a __stopping time__). 

The American premium is at least the European one because the American holder can always wait until expiration. The standard method to price American options is the Cox, Ross, Rubinstein (CRR) binomial model.

__Reference:__
* [Cox, J. C.; Ross, S. A.; Rubinstein, M. (1979). "Option pricing: A simplified approach". Journal of Financial Economics. 7 (3): 229. CiteSeerX 10.1.1.379.7582. doi:10.1016/0304-405X(79)90015-1](https://www.sciencedirect.com/science/article/pii/0304405X79900151)

### CRR Binomial Lattice Model
The CRR model assumes that the movement of the share price of the underlying asset can be modeled as a binomial tree, where at each time step, the price can either move up by a factor of $u$ or down by a factor of $d$. The probability of an up move is the __risk-neutral probability__ $q$, and the probability of a down move is $(1-q)$.

> __Risk-Neutral, what?__ In the CRR approach, the binomial tree parameters are no longer data driven. Models compute the $u$, $d$, and $q$ parameters. The risk-neutral probability $q$ is derived so that the expected share price grows at the risk-free rate $g_{y}$.

Let $S_{0}=S(0)$ denote the current share price (units: USD/share). One time step in the future, the expected share price under the risk-neutral pricing rule $\mathbb{Q}$ is given by:
$$
\begin{align*}
\mathcal{D}_{1,0}(g_{y})\cdot S_{0} & = \mathbb{E}_{\mathbb{Q}}\left(S_{1}\right) && \text{grow at the risk-free rate under }\mathbb{Q}\\
& = q\cdot(u\cdot S_{0}) + (1-q)\cdot(d\cdot S_{0}) && \text{up with probability } q\text{, down with } (1-q)\\
& = S_{0}\bigl(qu + (1-q)d\bigr) && \text{factor out } S_{0}\\
& = S_{0}\bigl(qu + d - qd\bigr) && \text{expand } (1-q)d\\
& = S_{0}\bigl(d + q(u - d)\bigr) && \text{collect the } q \text{ terms}
\end{align*}
$$
where $\mathcal{D}_{1,0}(g_{y})=e^{g_{y}\Delta t}$ is the risk-free accumulation factor for one time step of length $\Delta t$ (units: years). Dividing by the initial share price $S_{0}$ and solving for $q$ gives:
$$
\boxed{
\begin{align*}
q = \frac{\mathcal{D}_{1,0}\left(g_{y}\right) - d}{u - d}\quad\blacksquare
\end{align*}}
$$
This is the risk-neutral probability we found by ruling out arbitrage when we introduced lattice models, with $R_{f}=\mathcal{D}_{1,0}(g_{y})$.

However, unlike the data-driven binomial model we discussed earlier, the CRR model derives $u$ and $d$ based on the volatility $\sigma$ of the underlying asset, the time step $\Delta t$, and the symmetric condition $ud = 1$. Let $a=\ln u$, so $\ln d=-a$. Over one time step, the growth rate $g=\frac{1}{\Delta t}\ln\bigl(S(t+\Delta t)/S(t)\bigr)$ is $+a/\Delta t$ with the real-world probability $p$ and $-a/\Delta t$ with probability $1-p$. 

Each step is a Bernoulli trial: let $X=1$ for an up move and $X=0$ for a down move, so $\mathbb{E}_{\mathbb{P}}(X)=p$ and $\operatorname{Var}_{\mathbb{P}}(X)=p(1-p)$. In [L3b](../../week-3/L3b/CHEME-5660-L3b-Lecture-LatticeModels-Equity-Price-Fall-2026.ipynb), we used these two facts to find the mean and variance of the growth rate on the tree. With $\ln u=a$ and $\ln d=-a$, they become:
$$
\begin{align*}
\mathbb{E}_{\mathbb{P}}(g) & = \frac{p\ln u+(1-p)\ln d}{\Delta t} = \frac{(2p-1)\,a}{\Delta t} && \text{substitute } \ln u=a \text{ and } \ln d=-a\\
\operatorname{Var}_{\mathbb{P}}(g) & = \frac{p(1-p)}{\Delta t^{2}}\left[\ln\left(\frac{u}{d}\right)\right]^{2} = \frac{4p(1-p)\,a^{2}}{\Delta t^{2}} && \text{substitute } \ln(u/d)=2a
\end{align*}
$$
Under geometric Brownian motion, $\ln\bigl(S(t+\Delta t)/S(t)\bigr)=\mu_{g}\Delta t+\sigma\sqrt{\Delta t}\,Z$, so $g$ has mean $\mu_{g}$ and variance $\sigma^{2}/\Delta t$. Setting the tree's mean and variance equal to these values gives:
$$
\begin{align*}
\frac{(2p-1)\,a}{\Delta t} & = \mu_{g} && \text{match the mean growth rate}\\
\frac{4p(1-p)\,a^{2}}{\Delta t^{2}} & = \frac{\sigma^{2}}{\Delta t} && \text{match the variance}\\
\frac{a^{2}}{\Delta t^{2}} - \left(\frac{(2p-1)\,a}{\Delta t}\right)^{2} & = \frac{\sigma^{2}}{\Delta t} && \text{use } 4p(1-p) = 1-(2p-1)^{2}\\
\frac{a^{2}}{\Delta t^{2}} - \mu_{g}^{2} & = \frac{\sigma^{2}}{\Delta t} && \text{the term in parentheses is } \mu_{g} \text{ (first line)}\\
a^{2} & = \sigma^{2}\Delta t + \underbrace{\mu_{g}^{2}\Delta t^{2}}_{\text{small}} && \text{solve for } a^{2}
\end{align*}
$$
where $\mu_{g}$ is the mean growth rate (units: $\mathrm{yr}^{-1}$). For a small time step, the second term is small next to the first. Cox, Ross, and Rubinstein keep only the first term, so $a=\sigma\sqrt{\Delta t}$ and:
$$
\boxed{
\begin{align*}
u &= e^{\sigma \sqrt{\Delta t}} \\
d &= e^{-\sigma \sqrt{\Delta t}}
\end{align*}}
$$
where $\sigma$ is the volatility (units: $\mathrm{yr}^{-1/2}$). Dropping the second term removes $\mu_{g}$ from $a$. The mean equation ties the real-world probability to the mean growth rate, $p=\frac{1}{2}\bigl(1+\mu_{g}\Delta t/a\bigr)$, so $u$ and $d$ depend on neither $\mu_{g}$ nor $p$. They depend only on the volatility $\sigma$, which is the same under $\mathbb{P}$ and $\mathbb{Q}$. [Want more on where this comes from?](advanced/crr_factors/CHEME-5660-L9b-Advanced-CRR-Factors-Fall-2026.ipynb) Let's use this lattice to price American contracts.

### Decision-Rule for American Options
The ability to exercise early introduces an additional decision-making layer in the pricing process beyond that of European options. 

>__Decision Rule__: At each node in the binomial tree, the holder of an American option must decide whether to exercise the option immediately or to hold onto it for potential future gains. The decision is based on comparing the immediate exercise value with the expected value of holding the option discounted back to the present value.

Let's look at a schematic of the decision rule for an American option contract.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L9b-American-Decision-Rule.svg" width="700" alt="Left, a three-step binomial lattice from t = 0 (now) to t = 3 that starts at the share price S0. The first step is labeled (u, q) for an up move and (d, 1 − q) for a down move, and the prices at t = 3 are u³S0, u²dS0, ud²S0, and d³S0. The top cell from t = 2 to t = 3 is shaded and enlarged in a panel on the right: the node value V̂(2,2) branches to V̂(3,3) with (u, q) and to V̂(3,2) with (d, 1 − q). Below the panel is the decision rule for an American contract: the node value is the larger of the payoff V(K, S) from exercising now and the discounted risk-neutral expectation of the two next values from holding for the future."/>
    </center>
</div>

At expiration, each node's value is the contract's payoff. Working backward, the American holder takes the larger of exercising now and holding for the future, while the European holder can only hold:
$$
\boxed{
\begin{align*}
\hat{V}^{\text{AM}}_{t,k} & = \max\Bigl\{\underbrace{V(K,S_{t,k})}_{\text{exercise now}},~\underbrace{\mathcal{D}^{-1}_{t+1,t}(g_{y})\cdot\overbrace{\bigl(q\,\hat{V}^{\text{AM}}_{t+1,k+1} + (1-q)\,\hat{V}^{\text{AM}}_{t+1,k}\bigr)}^{\mathbb{E}_{\mathbb{Q}}}}_{\text{hold for the future}}\Bigr\}\\
\hat{V}^{\text{EU}}_{t,k} & = \mathcal{D}^{-1}_{t+1,t}(g_{y})\cdot\overbrace{\bigl(q\,\hat{V}^{\text{EU}}_{t+1,k+1} + (1-q)\,\hat{V}^{\text{EU}}_{t+1,k}\bigr)}^{\mathbb{E}_{\mathbb{Q}}}
\end{align*}}
$$
where $\hat{V}_{t,k}$ is the contract's value at the node with $k$ up moves in $t$ steps, $S_{t,k}=S_{0}u^{k}d^{t-k}$ is the share price there, and $\mathcal{D}^{-1}_{t+1,t}(g_{y})=e^{-g_{y}\Delta t}$ discounts one time step. The premium is the value at the root, $\mathcal{P}=\hat{V}_{0,0}$. The difference $\mathcal{P}^{\text{AM}}-\mathcal{P}^{\text{EU}}\geq 0$ is the value of the right to exercise early, the __early-exercise premium__.

Let's outline some pseudocode for pricing an option contract on the CRR lattice.

__Initialize__: Given the share price $S_{0}$, the strike price $K$, the payoff function $V(K,S)$ of the call or put, the time to expiration $T$ (units: years), the volatility $\sigma$, the risk-free rate $g_{y}$, and the number of time steps $h$. Compute the time step $\Delta t = T/h$, the up and down factors $u$ and $d$, the risk-neutral probability $q$, and the one-step discount factor $\mathcal{D}^{-1}_{t+1,t}(g_{y})=e^{-g_{y}\Delta t}$.

1. Build the share price lattice: $S_{t,k} \gets S_{0}u^{k}d^{t-k}$ for each level $t = 0,1,\dots,h$ and node $k = 0,1,\dots,t$.
2. At expiration, set each node value to the payoff: $\hat{V}_{h,k} \gets V(K,S_{h,k})$ for $k = 0,1,\dots,h$.
3. For each level $t = h-1, h-2, \dots, 0$ __do__:
   - For each node $k = 0,1,\dots,t$ __do__:
      - a. Compute the value of holding for the future: $H_{t,k} \gets \mathcal{D}^{-1}_{t+1,t}(g_{y})\cdot\bigl(q\,\hat{V}_{t+1,k+1} + (1-q)\,\hat{V}_{t+1,k}\bigr)$.
      - b. If the contract is American, __then__: $\hat{V}_{t,k} \gets \max\bigl\{V(K,S_{t,k}),~H_{t,k}\bigr\}$. If the contract is European, __then__: $\hat{V}_{t,k} \gets H_{t,k}$.
4. The premium is the value at the root: $\mathcal{P} \gets \hat{V}_{0,0}$.

Before we dig into the implementation on real data, let's look at a simple hypothetical example from Chapter 13 of Hull that compares an American and European option contract price.

> __Example:__ Consider a put contract. The contract's strike price is $K$ = 52 USD/share on a stock whose current price is $S_{0}$ = 50 USD/share. There are two time steps of 1 year each. In each period, the stock price moves up by a factor of $u$ = 1.2 or down by a factor of $d$ = 0.8. Let the continuously compounded risk-free rate be $g_{y}$ = 5%, so the risk-neutral probability of an up move is $q=(e^{0.05}-0.8)/(1.2-0.8)$ = 0.6282.

What is the price of the American and European put option contracts?

<div>
    <center>
        <img class="course-diagram" src="figs/Fig-HullExample-American-v-European-Schematic.svg" width="640" alt="Hull's two-step put tree with K = 52, S0 = 50, u = 1.2, and d = 0.8, American contract on the left and European on the right. At the down node at t = 1 (share price 40), the American holder exercises for 12 instead of holding for 9.46, so the premiums at the root are 5.09 and 4.19 USD/share."/>
    </center>
</div>

At the down node at $t=1$, exercising pays 12 USD/share, while holding is worth 9.46 USD/share, so the American holder exercises. That one decision raises the premium from 4.19 to 5.09 USD/share, an early-exercise premium of about 0.90 USD/share.

Let's compute every node of this lattice, then apply the same method to contracts from the course's options archive:

> __Examples:__
>
> __Review on your own:__ [▶ Let's compute the Premium for Hull's Hypothetical American Put using a binomial lattice](CHEME-5660-L9b-Example-Hull-American-Put-Fall-2026.ipynb). In this example, we'll compute the price of a hypothetical American put option contract using backward induction over a binomial lattice model, and compare our results to the same contract written as a European contract.
>
> __We'll work through in lecture:__ [▶ Let's compute the Premium for Market-Quoted American Call and Put Options using CRR](CHEME-5660-L9b-Example-AmericanContractsPremium-Fall-2026.ipynb). In this example, we'll compute the price of American call and put option contracts using the CRR binomial model, compare our results to observed market prices, and measure each contract's early-exercise premium.
___

## Optional Advanced Material

The [advanced index](advanced/README.md) describes the optional notebooks below:

* [▶ Contingent claims and the Black–Scholes–Merton formulas](advanced/contingent_claims/CHEME-5660-L9b-Advanced-ContingentClaims-BSM-Fall-2026.ipynb). In this notebook, we explore the theoretical foundations of option pricing. We find the risk-neutral probability on a one-step lattice and price event contracts with state prices. We then derive the BSM equation and build the call premium from a digital claim and an asset-or-nothing claim. Finally, we compare a client's certainty equivalent for a one-dollar bet with its market price to find where the risk premium goes. It is a derivation notebook with no code.

* [▶ SPXW volatility skew](advanced/spxw_volatility_skew/CHEME-5660-L9b-Advanced-SPXW-Volatility-Skew-Fall-2026.ipynb). In this notebook, we analyze the volatility skew of SPXW options. We use an end-of-day chain of SPXW options, the European, cash-settled S&P 500 index options with weekly expirations. Put–call parity gives the discount factor and forward index level, and we back out the implied volatility at each strike near the forward. We then compare the resulting skew with SPY's realized volatility.

* [▶ Deriving the CRR up and down factors](advanced/crr_factors/CHEME-5660-L9b-Advanced-CRR-Factors-Fall-2026.ipynb). In this notebook, we derive the up and down factors of the CRR model by matching the mean and variance of the lattice's one-step growth rate to those of geometric Brownian motion. We then show that the lattice still matches geometric Brownian motion when we price with the risk-neutral probability, and find how small the time step must be to keep that probability between zero and one. It is a derivation notebook with no code.

___

## Summary
This lecture priced European options with the Black–Scholes–Merton formulas and explored how American options are priced using the CRR binomial model.

> __Key Takeaways:__
>
> * __The Black–Scholes–Merton formulas price European options:__ We computed European call and put premiums in closed form, and they match the risk-neutral premium that Monte Carlo simulation estimates. We backed out the implied volatility from a market premium by bisection.
> * __Binomial trees price American options:__ The CRR model creates a discrete price lattice where at each node, the holder decides whether to exercise immediately or hold the option, making backward induction necessary to compute prices.
> * __The decision rule sets the early-exercise premium:__ We priced Hull's two-step put with the American and European rules on the same lattice. The American holder exercises at the one node where exercising pays more than holding, and that decision accounts for the whole difference between the two premiums.

The early-exercise premium is what the right to exercise before expiration is worth, and the binomial lattice computes it.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___